# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Uman-66/Flyrank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I am working in the **Refresh / Content Opportunity Scoring** lane. I frame this as a **scoring and ranking** task because the goal is to assign each page an opportunity score and then rank pages from highest to lowest priority for human review.

The output is not intended to automatically decide that a page must be refreshed. Instead, it should help a content or SEO team decide which pages deserve attention first when review time is limited. A ranking is therefore more useful than a simple yes/no prediction because it supports prioritization.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm the ML task framing

task_type = "Scoring / Ranking"
lane = "Refresh / Content Opportunity Scoring"

print("Lane:", lane)
print("ML task type:", task_type)
print("Intended output: ranked page-level opportunity scores")

Lane: Refresh / Content Opportunity Scoring
ML task type: Scoring / Ranking
Intended output: ranked page-level opportunity scores


## 2. Target or proxy

For the initial version of this project, I will use **`trend_direction`** as the observed outcome signal for identifying pages with a declining trend.

The target information comes from the observed search-performance trend in the starter dataset rather than from a future outcome. In particular, pages with a `trend_direction` of `down` represent pages showing an observed declining trend.

This is therefore a **provisional proxy**, not a future-looking prediction target. I will not claim that the model predicts future page decline.

A stronger future version could define a future-looking target, such as using signals from an earlier time window to predict whether a page's performance declines during a later time window. I would also need to make sure that features do not contain the same information used to construct the target, in order to avoid target leakage.

In [11]:
# Inspect the observed trend signal used as the provisional proxy

target_column = "trend_direction"

print("Target/proxy:", target_column)

print("\nObserved trend values:")
print(df[target_column].value_counts())

print("\nObserved trend proportions:")
print(df[target_column].value_counts(normalize=True))

Target/proxy: trend_direction

Observed trend values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Observed trend proportions:
trend_direction
down      0.542067
stable    0.198733
up        0.146267
new       0.074533
flat      0.038400
Name: proportion, dtype: float64


## 3. Success metric

My primary success metric will be **Precision@50**.

Precision@50 measures how many of the top 50 pages in the model's ranking are labelled as declining according to the provisional target.

The metric is appropriate because the intended action is to create a prioritized review queue. If a content or SEO team can review only a limited number of pages, the quality of the highest-ranked pages matters more than simply getting the overall classification correct.

A higher Precision@50 means that a larger proportion of the first 50 recommended pages are relevant according to the provisional target.

The eventual ML approach should also be compared with simple fixed-rule baselines. ML is only worthwhile if its ranking provides useful improvement over simpler approaches.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the primary evaluation metric

K = 50

print("Primary success metric: Precision@50")
print("K:", K)
print("Interpretation: proportion of relevant pages among the top 50 ranked pages")

Primary success metric: Precision@50
K: 50
Interpretation: proportion of relevant pages among the top 50 ranked pages


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one page**.

Each row represents one page-level observation, while the columns contain observable signals about that page, such as search impressions, CTR, position, content freshness, and the provisional declining label.

This unit matches the decision because the eventual output will rank individual pages for possible human review.

The dataframe below shows a small slice of the actual starter dataset. No client names, private queries, or page URLs are needed for this analysis.

In [13]:
# Show the unit of analysis as a real dataframe
# One row = one page

print("One row = one page-level observation")

page_sample = df[
    [
        "content_id",
        "impressions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update",
        "trend_direction"
    ]
].head(10)

display(page_sample)

One row = one page-level observation


,content_id,impressions_90d,ctr,avg_position,days_since_last_update,trend_direction
0,content_304f48230142,3803,0.76,10.6,20,down
1,content_a1fb4e703a9e,15320,0.05,20.3,25,down
2,content_9aa793d4d895,12581,0.09,36.5,20,down
3,content_331d6c4de07b,11751,0.49,6.2,22,stable
4,content_d99b7a2d90ca,19140,0.13,44.0,14,down
5,content_d4084a4bc775,3970,0.03,8.5,20,down
6,content_9a34b442b552,20,0.00,7.0,20,down
7,content_a63219c6e95a,1724,0.06,21.2,22,stable
8,content_5e6c160719bc,32574,0.09,46.0,20,down
9,content_c27558df2b0c,1240,0.16,4.9,104,down


## 5. Why ML beats a fixed rule here

A simple fixed rule could say that a page should be prioritized if it has not been updated for more than a certain number of days.

For example, a rule such as `days_since_last_update > 180` would treat all pages older than 180 days in the same way. However, content age alone may not capture the full situation of a page.

A page can be old but still perform well, while a more recently updated page could show other signals that make it worth reviewing. Multiple signals such as impressions, CTR, position, content freshness, and observed trend may provide a more useful picture when considered together.

ML may therefore be useful because it can combine several observable signals and produce a ranked score rather than relying on one manually chosen threshold.

However, I will not assume that ML is automatically better. The eventual model should be compared against simple fixed-rule baselines, and the additional complexity should only be justified if it provides useful improvement for the intended decision.

In [14]:
# Example of a simple fixed-rule baseline

RULE_THRESHOLD = 180

rule_priority_count = (
    df["days_since_last_update"] > RULE_THRESHOLD
).sum()

print(
    f"Pages with more than {RULE_THRESHOLD} days since last update: "
    f"{rule_priority_count:,}"
)

print("\nThis represents a simple rule-based baseline.")
print("A future ML scoring approach should be compared against simple baselines like this.")

Pages with more than 180 days since last update: 174

This represents a simple rule-based baseline.
A future ML scoring approach should be compared against simple baselines like this.


## Self-check

- [x] Every section is filled with markdown thinking and supporting code.
- [x] The lane is Refresh / Content Opportunity Scoring.
- [x] The ML task type is defined as scoring/ranking.
- [x] The provisional proxy is `trend_direction`, with `down` representing an observed declining trend.
- [x] The target is described as an observed outcome proxy, not future prediction.
- [x] Precision@50 is defined as the primary success metric.
- [x] The unit of analysis is clearly defined as one row = one page.
- [x] A real dataframe from the starter dataset is displayed.
- [x] The output is connected to a real content/SEO review action.
- [x] The reason for using ML instead of a fixed rule is explained.
- [x] Simple fixed-rule baselines will be considered.
- [x] No client names, private queries, or sensitive page information are intentionally exposed.
- [x] Claims use careful language such as observed, measured, directional, and decision-support.
- [x] The notebook should be run top to bottom before submission.
- [x] The executed notebook should be committed under `work/notebooks/w02_ml_task_framing.ipynb`.